
## 2. Method 1: Spectral Hashing
### 2.1 The Context

* **Open Modification Searching (OMS)** is used to identify *post-translationally modified* peptides in mass spec data.
  * **Need to define PTMs briefly**
* Traditional proteomics search algorithm attempt to match fragmentation spectra by first identifying the mass of the intact peptide that was isolated to generate a given fragmentation spectrum. 
  * Then, they only consider peptides from the peptide database with a matching mass. 
  * This strategy greatly reduces the search space, taking a list of hundreds of thousands of peptides in the database and reducing it to dozens of peptides.
* In **OMS**, every peptide in the database is considered as a potential match to every MS2 fragmentation spectrum. 
  * When a peptide-spectrum match is found, you can look for *mass shifts*. By calculating the difference between the mass of the matched peptide from the database and the mass of the precursor in the MS1 data, you can determine if the mass was shifted. Any mass shifts are caused by the addition of a  post-translational modifications (PTMs) 
  * "All possible modifications are implicitly considered."
* OMS is **inefficient** because we have to compare every spectrum against *every peptide in the database*. ~~search window is significantly larger than the standard search window.~~
* The solution? Instead of comparing every spectrum to every peptide, you can use a clustering algorithm to identify groups of similar spectra. 
  * If you know the identity of some of the spectra ahead of time (e.g., those that can be identified using a traditional search), then you have a pretty good idea of the identity of every peptide in the cluster.  
  * This approach was used by [ANN-SoLo](https://pubs.acs.org/doi/full/10.1021/acs.jproteome.9b00291)

<br></br>
<center><img src="ANN-SoLo-Graphical-Abstract.jpeg" width="685" height="400"></center>


### 2.2 Conceptual Introduction

**What is clustering, and why do we need it?**

Recall the OMS problem from section 2.1. We must compare every unknown spectrum against every peptide in the database, which comes out to be millions of comparisons. Clustering offers a much smarter approach.

Let me introduce an analogy. Suppose you're building a large Lego, with about ~2000 pieces. Here are two ideas for how we could go about doing this.

1. The naive approach:
    * Dump all of the pieces out at once. 
    * Open up your instruction manual
    * For each instruction, find the necessary piece(s) from your pile of lego pieces
2. Clustering approach: 
    * Dump all of the pieces out at once.
    * Make distinct groups of all of the same pieces, not necessarily distinct by color
    * Open up your instruction manual
    * For each instruction, look at each of your piles. If we grab a single lego piece from any pile, it is, by construction, representative of all legos in that pile. So, if I find a 2x4x1 studded lego, then I know all legos in that pile are also 2x4x1 studs. This allows me to efficiently choose another pile if this is not what I am looking for.

Note
 * What if there were 2,000,000 lego pieces? 
 * There exist several ways to cluster legos, many of which I've tried, which is also incidentally why we need a well defined way to compute relative distances (differences here) between two objects. There is a balance to be struck between "too small/specific of a pile" and "too large/generic of a pile" and it is wholly dependent on the contents of the lego set.

For spectra, clustering works the same way:
* **Group similar spectra together** based on their characteristics
* **Identify just a few representative spectra** in each group (using traditional search methods)
* **Assign the same identity to all spectra in that group** 

This transforms millions of comparisons into just thousands!

**So how do we group spectra together?**

We can:
1. **Convert each spectrum into a vector** (a list of numbers that captures the spectrum's characteristics)
2. **Measure distances between vectors** — spectra with similar characteristics end up close together in multi-dimensional space
3. **Group nearby vectors into clusters** — this is the actual "clustering"
4. **Match by association** — unknown spectra inherit identities from known spectra in their cluster

**A visual example in 2D**

The graphics below walk through a simplified example where we cluster spectra in just 2 dimensions. In reality, we'll work with hundreds of dimensions, but the core idea is exactly the same.

The key insight from these graphics: **distance = similarity**. When two spectra map to nearby points in space, they're similar. When they're far apart, they're different.

<br></br>
<img src="Clustering-1.jpeg" width="500" height="675">
<img src="Clustering-2.jpeg" width="500" height="675">

### 2.3 From Peaks to Vectors

#### Understanding "high-dimensional" data

When you look at at some random spectrum, you will often see data in 2 dimensions (we'll ignore retention time for now): m/z along the x axis, and intensity along the y axis. Now imagine, that each m/z value was actually it's own "dimension". That is, we made a list mapping m/z to intensity, component-wise.

Let's break this down with a concrete example. Imagine we divide the m/z range into bins:
* Bin 1: m/z 100-101 (intensity = 0)
* Bin 2: m/z 101-102 (intensity = 500)
* Bin 3: m/z 102-103 (intensity = 0)
* Bin 4: m/z 103-104 (intensity = 1200)
* ... and so on for hundreds or thousands of bins

We can represent this spectrum as a vector: `[0, 500, 0, 1200, ...]` where each position corresponds to one m/z bin. If we have 880 bins (m/z defined from 1 to 880), that's an 880-dimensional vector! Each dimension tells us "how much intensity is in this particular m/z range?"

This is what we mean by "high-dimensional." Dimension in this case is simply the length of this list, or vector. Not that the spectrum itself is hard to visualize, as is often the case in >3 dimensions, but that mathematically, we're representing it as a point in a space with hundreds of dimensions. This allows us to easily compare it to other points in space with the same dimensions.

#### The challenge of dimensionality

Now here's the problem: if we want fine precision in our m/z measurements (say, bins of 0.01 Da), we might need 880,000 dimensions. That's extremely computationally expensive for clustering algorithms.

#### The solution: spectral embedding

Spectral embedding is the process of taking our high-dimensional representation and compressing it down to a lower-dimensional space (like say 500 dimensions instead of 880,000) while preserving the important relationships between spectra.

#### What's a hash function?
A **hash function** is a deterministic algorithm that converts input data of any size into a fixed-size output (called a "hash" or "digest"). Hash functions have several key properties that make them useful:

#### Key Properties:
1. **Deterministic**: The same input always produces the same output
   - `hash(129.103)` → always returns `12910`
   - `hash(129.103)` → always returns `12910` (same result every time)

2. **Fixed output size**: No matter the input size, the output is always the same length
   - `hash("short")` → `a1b2c3d4` (8 characters)
   - `hash("this is a much longer input")` → `e5f6g7h8` (also 8 characters)

3. **Efficiently computable**: Fast to calculate, even for large inputs



#### How we'll do this: Feature Hashing

The specific technique we'll use is called **feature hashing**:
* Start with a spectrum as m/z–intensity pairs
* Create fine-grained bins (0.01 Da width) to capture precise mass information
* Use a hash function to map these many bins into fewer "hash buckets"
* Accumulate intensities in each bucket

**Why this works:**
* Compact representation: 800 dimensions instead of 880,000
* Preserves similarity: similar spectra still produce similar vectors
* Fast computation: enables efficient clustering and searching

The trade-off is that some different m/z bins might hash to the same bucket (a "collision"), but we'll see that the algorithm handles this gracefully without losing too much information.

In the code sections below, we'll implement this step-by-step and demonstrate that similarity is indeed preserved.

**Theoretical steps**

The paper simplifies feature hashing down to two main steps:

1. Convert the spectrum to a sparse vector
using small mass bins to tightly capture
fragment masses.
2. Hash the sparse, high-dimensional
vector to a lower- dimensional vector by using a hash
function to map the mass bins to a limited number of hash bins

![fig-1.png](https://pubs.acs.org/cms/10.1021/acs.jproteome.9b00291/asset/images/medium/pr9b00291_0001.gif)

I explain what this means practically, line by line, in the code below.

In [ ]:

# @title Run this cell to import all necessary packages

%pip install matplotlib

import spectrum_utils.plot as sup
import spectrum_utils.spectrum as sus
import pyteomics
from pyteomics import mzml, auxiliary
import plotly.io as pio
import plotly.tools as tls
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from rapidhash import rapidhash
from IPython.display import display, Latex
from util import *
from matplotlib.lines import Line2D


Throughout our examples, we'll use two mzml files:

In [ ]:
# @title Run this to define our mzml paths
 
snippet_mzml_path = '04-17-23_CA_Tryp_HCD_10min_CLEAN.mzML' 
"""
Smaller file, contains a snippet of the data from the full version below
"""
full_calibrated_mzml_path = '04-17-23_CA_Tryp_HCD_10min_CLEAN-calib.mzML'
"""
Full, sample mzML file
"""

The get_MS2_object function takes in a path and a scan number. This scan number tells is to retrieve the MS2 corresponding to the scan number from some mzML file.

In [ ]:
ms2_spectrum_5672 = get_MS2_object(snippet_mzml_path, 5672)
plot_MS2(ms2_spectrum=ms2_spectrum_5672)

Grab the intensity and mz arrays from the object above:

In [ ]:
# grab mz values from that ms2 object as a list
spectrum_mz = ms2_spectrum_5672.mz

# grab intensity from that ms2 object as a list
spectrum_int = ms2_spectrum_5672.intensity

Here are the numerical values AKA how we represent our list of peaks:
```python
spectrum_mz =
[129.10374451 147.11447144 185.16696167 186.17019653 213.16217041
 229.1210022  260.20001221 298.14343262 300.1590271  328.190979
 347.23321533 348.23629761 462.26141357 463.26467896 484.20996094
 575.34680176 646.3848877  647.38745117 743.4017334  761.41223145
 762.41516113 763.41802979 874.49749756 875.50030518]


spectrum_int =
[2174.818  3145.84   9230.925  2883.7764 4380.564  2282.9873 3063.1228
 2072.459  3142.8696 2672.6929 5380.33   2240.3975 6059.931  2831.662
 2237.5989 3632.9626 4356.975  2494.9565 2366.8223 8133.1064 5098.785
 2226.6924 4098.929  2766.6738]
```

In [ ]:
# Intensity and mz values are parallel arrays, which you can either count the lengths of 
# or just assert and see it doesn't throw an error
assert(len(spectrum_int) == len(spectrum_mz))

Speaking of which, take a look at the spectrum graph, figure, and theoretical steps above again! Here are some ideas we can find from them:
- This particular spectrum ranges from mz values from 0 through ~880.
- We CAN make a list of 

We'll rewrite this into our new array of length 880 as a numerical example.
*   In the 129th idx: 2174.818
*   In the 147th idx: 3145.84
*   In the 185th idx: 9230.925
*   And so on...


**Why use more precise m/z indices?**

If we had 200 unique m/z values ranging from 100–300 and rounded each to the nearest integer, we’d quickly run into overlaps. For example, every value between 128.5 ≤ m/z < 129.5 would collapse into the same index: 129. That means different peaks could get “bucketed” together, and we'd lose resolution.

To avoid this, we can create finer-grained buckets:
* Nearest 1.0 → array length = max(mz)
* Nearest 0.1 → array length ≈ max(mz) * 10
* Nearest 0.001 → array length ≈ max(mz) * 1000

So if max(mz) = 880, then:
* 0.1 precision → 8,800 buckets
* 0.001 precision → 880,000 buckets




<!-- \text{Here's an empty list: } &\quad [\ ] \\[0.5em]-->
### Accurate vs. Naive approach

Here's the Naive approach with making such a vector. Here, we round down each mz to the nearest integer.
$$
\text{Approach 1: Rounding to nearest integer (0.1 Da bins)} \\[0.5em]
\begin{align}
\text{Here's a list with 880 zeroes filled in: } &\quad [\underset{0}{0}, \underset{1}{0}, \underset{2}{0}, \ldots, \underset{879}{0}, \underset{880}{0}] \\[0.5em]
\text{Now, let's add a sample of such values at each index:} & \quad [\underset{0}{0}, \ldots, \underset{129}{2174.818}, \ldots, \underset{147}{3145.84}, \ldots, \underset{880}{0}] 
\end{align}
$$

Like we mentioned though, this takes a lot of accuracy away, and results in quite a few collisions. The approach using finer (to the nearest 0.01 mz) buckets:

$$
\text{Approach 2: Rounding to nearest 0.01 Da (higher precision)} \\[0.5em]
\begin{align}

\text{Here's a list with 88,000 zeroes filled in: } &\quad [\underset{0}{0}, \underset{1}{0}, \underset{2}{0}, \ldots, \underset{87999}{0}, \underset{88000}{0}] \\[0.5em]
\text{Now, let's add a sample of such values at each index:} & \quad [\underset{0}{0}, \ldots, \underset{12910}{2174.818}, \ldots, \underset{14711}{3145.84}, \ldots, \underset{88000}{0}] 
\end{align}
$$

In [ ]:
# @title Run this to see the difference in collisions between widths

def count_collisions(mz_array, bin_width):
    """
    Count how many peaks collide (fall into the same bin) for a given bin width.
    
    Returns: (num_collisions, num_peaks)
    """
    bin_ids = np.floor(np.asarray(mz_array) / bin_width).astype(np.int64)
    buckets = {}
    for i, b in enumerate(bin_ids):
        buckets.setdefault(b, []).append(i)
    
    collision_count = sum(len(idxs) for idxs in buckets.values() if len(idxs) > 1)
    return collision_count, len(mz_array)


def analyze_per_spectrum_collisions(mzml_path, bin_widths=(1.0, 0.01), max_spectra=None):
    """
    Analyze collision rates for each individual MS2 spectrum in an mzML file.
    
    Parameters
    ----------
    mzml_path : str
        Path to the mzML file
    bin_widths : tuple
        Bin widths to compare (default: 1.0 Da and 0.01 Da)
    max_spectra : int or None
        Limit number of spectra to analyze (None = all)
    
    Returns
    -------
    DataFrame with collision statistics per spectrum
    """
    all_spectra = get_all_MS2_objects(mzml_path=mzml_path)
    
    if max_spectra:
        all_spectra = all_spectra[:max_spectra]
    
    results = []
    for i, ms2 in enumerate(all_spectra):
        row = {'spectrum_idx': i, 'num_peaks': len(ms2.mz)}
        
        for width in bin_widths:
            collisions, total = count_collisions(ms2.mz, width)
            row[f'collisions_{width}Da'] = collisions
            row[f'collision_rate_{width}Da'] = collisions / total if total > 0 else 0
        
        results.append(row)
    
    df = pd.DataFrame(results)
    return df, all_spectra


# Analyze collisions per spectrum
mzml_file = full_calibrated_mzml_path
collision_df, all_objs = analyze_per_spectrum_collisions(mzml_file, max_spectra=None)

# Summary statistics
print(f"=== Per-Spectrum Collision Analysis ===")
print(f"File: {mzml_file}")
print(f"Total spectra analyzed: {len(collision_df)}")
print(f"\n--- Bin Size = 1.0 Da ---")
print(f"  Spectra with collisions: {(collision_df['collisions_1.0Da'] > 0).sum()} / {len(collision_df)}")
print(f"  Mean collisions per spectrum: {collision_df['collisions_1.0Da'].mean():.2f}")
print(f"  Max collisions in a spectrum: {collision_df['collisions_1.0Da'].max()}")
print(f"  Mean collision rate: {collision_df['collision_rate_1.0Da'].mean()*100:.2f}%")

print(f"\n--- Bin Size = 0.01 Da ---")
print(f"  Spectra with collisions: {(collision_df['collisions_0.01Da'] > 0).sum()} / {len(collision_df)}")
print(f"  Mean collisions per spectrum: {collision_df['collisions_0.01Da'].mean():.2f}")
print(f"  Max collisions in a spectrum: {collision_df['collisions_0.01Da'].max()}")
print(f"  Mean collision rate: {collision_df['collision_rate_0.01Da'].mean()*100:.2f}%")

# Visualization: Distribution of collision counts per spectrum
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: 1.0 Da bin collisions
axes[0].hist(collision_df['collisions_1.0Da'], bins=30, alpha=0.7, color='coral', edgecolor='black')
axes[0].axvline(collision_df['collisions_1.0Da'].mean(), color='red', linestyle='--', 
                linewidth=2, label=f"Mean: {collision_df['collisions_1.0Da'].mean():.1f}")
axes[0].set_xlabel('Number of Collisions per Spectrum')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Collisions per Spectrum (Bin = 1.0 Da)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: 0.01 Da bin collisions
axes[1].hist(collision_df['collisions_0.01Da'], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
axes[1].axvline(collision_df['collisions_0.01Da'].mean(), color='darkblue', linestyle='--', 
                linewidth=2, label=f"Mean: {collision_df['collisions_0.01Da'].mean():.1f}")
axes[1].set_xlabel('Number of Collisions per Spectrum')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Collisions per Spectrum (Bin = 0.01 Da)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

This clearly demonstrates why we need a smaller bin size. However, we can't just make bins arbitrarily small, we must choose a size that is both fine enough to preserve information and realistic for mass spectrometer accuracy.

The key consideration is that mass spectrometers have physical precision limitations. While a spectrometer might output a value like $ m/z =  10.023130 $, it cannot actually distinguish measurements to that level of precision. In practice, mass spectrometers are reliably accurate to approximately two decimal places (0.01 Da).

In [ ]:
WIDTH_OF_BIN = 0.01
LENGTH = int(max(spectrum_mz) // WIDTH_OF_BIN +1) # integer rounding up one.
print(f"We have to fit our m/zs into bins by rounding them to the nearest {WIDTH_OF_BIN}")
print(f"The length of our (simulated) array will be {LENGTH}")


### STEP 1

**Convert the spectrum to a sparse vector using small mass bins to tightly capture fragment masses.**

First, we'll define a function to convert m/z values to their "binned" index value.
For example: 129.103 m/z → index 12910

In [ ]:
# @title Conversion

def to_idx(mz):
    """
    Convert m/z value to a sparse vector index.
    """
    # concretely: 129.103 // 0.01 == 12910
    return int(mz // WIDTH_OF_BIN)

### Sparse arrays

With increased precision, we have bigger arrays. At very high precision (like 88K slots), things become computationally expensive and impractical. 

So our question now is, can we take that increased accuracy AND make it somehow smaller? 


- Defn - _Sparse Array_: An array of mostly zeroes, with a few values intermingled throughout the array
    - Here, we have a sparse array. Eg. indices 0 to 12909 are all 0. 
        - Why do we need all those indices? Can you think of a data structure that'd work better?




**Why use a map (dictionary) instead of an array?**

A **dictionary** (or "map") solves this elegantly:
- We only store the indices that actually have peaks
- Looking up a value by its index is still fast (O(1) on average)
- We skip over all the zeros entirely when iterating

This is the standard way to represent sparse vectors in practice. Instead of:
```
[0, 0, 0, ..., 2174.818, 0, 0, ..., 3145.84, 0, 0, ..., 0]  # 88,000 elements!
```

We store:
```
{12910: 2174.818, 14711: 3145.84, ...}  # Only 24 entries in this spectrum.
```

>The dictionary **keys** are our binned m/z indices, and the values are the corresponding **intensities**.

In [ ]:
# @title Create the sparse representation: map from m/z indices to intensities

mz_intensity_map = {}
# Populate the sparse vector with our spectrum data
for i, mz in enumerate(spectrum_mz):
    # Convert each m/z to its corresponding index
    mz_index = to_idx(mz)
    # Store the intensity
    mz_intensity_map[mz_index] = spectrum_int[i]

What does this look like?

In [ ]:
# @title Run to visualize the dictionary 
print("The map dictionary looks like:\n")
print(f"{'m/z bin':<12} {'Intensity':<12} {'Original m/z'}")
print("-" * 40)
for i, (idx, intensity) in enumerate(sorted(mz_intensity_map.items())):
    print(f"{idx:<12} {intensity:<12.3f} ({spectrum_mz[i]:.4f})")

print("\n" + "-" * 40)
print(f"Total entries: {len(mz_intensity_map)} (instead of ~88,000 zeros!)")

We're finished with the entirety of step 1, including the sparse vector optimization. 

### STEP 2 

**Hash the high-dimensional sparse vector to a lower-dimensional vector using a hash function**

In [ ]:
# @title Open this cell for Step 2

hash_buckets = 800  # Our target dimensionality (much smaller than ~880k sparse indices)

def hasher(num: int) -> int:
    """
    Hash function that maps sparse indices to a fixed number of buckets.

    Input: Large sparse index (e.g., 12910)
    Output: Small bucket index (0 to 399)
    """
    # Convert integer to bytes for hashing (rapidhash expects byte input)
    byte_representation = int(num).to_bytes(8, 'little')

    # Hash and mod to get bucket index in range [0, hash_buckets-1]
    return rapidhash(byte_representation) % hash_buckets

# Initialize our final feature 
# This is our compact, fixed-size representation of the spectrum
hash_matrix = [0] * hash_buckets  # Start with all zeros

# The final step: populate the hash buckets with intensities
# Multiple sparse indices may hash to the same bucket (collisions)
# We handle collisions by ADDING intensities (preserving total signal)

# see how map allows us to not have to look through every bin?
for sparse_idx, intensity in mz_intensity_map.items(): 
    # Get the hash bucket for this sparse index
    bucket_idx = hasher(sparse_idx)

    # Add the intensity to the bucket (handles collisions by summation)
    hash_matrix[bucket_idx] += intensity

print("Final spectral hash vector:")

end = "["
for hashedVals in hash_matrix:
    if hashedVals != 0:
        end += f"{hashedVals:.2f},"
    if hashedVals ==0:
        end += f"{hashedVals},"
end = end[:len(end)-1]
print(end,end="]\n")
print("Non-zero buckets:", sum(1 for x in hash_matrix if x > 0), f"out of {hash_buckets} buckets" )

**SUMMARY** of the spectral hashing process:
1. Started with: 24 (m/z, intensity) pairs
2. Essentially to: ~88,000 possible sparse indices, only 24 actually used
3. Hashed down to: 800 compact feature dimensions
4. Result: Fixed-size vector that preserves spectral similarity for ML/clustering

What does it mean to **preserve similarity?**

Maintaining the relative closeness between spectra after hashing, so you can perform clustering or nearest neighbor search without needing the full 88,000-dimensional sparse vector.

Two vectors $\vec{x}$ and $\vec{z}$ are more similar than $\vec{x}$ and $\vec{y}$ if their dot product, is greater than the latter.

$\vec{x} \cdot \vec{z} > \vec{x} \cdot \vec{y} $


Preserving similarity across your spectra means that once you hash it, it should still maintain about the same dot product across all values as when you bin it. 

Of course, there'll be a few more inconsistencies as you're going from 8800 $\rightarrow$ 800 dimensions, but it should be relatively the same.

In [ ]:
# Demonstrate similarity preservation between original sparse maps and hashed vectors
# Let's get multiple spectra and compare their similarities
from functools import lru_cache
# Get all spectra first

spectra_to_compare = get_all_MS2_objects("04-17-23_CA_Tryp_HCD_10min_CLEAN.mzML")

# Convert each spectrum to sparse map and hash vector representations
sparse_maps = []
hash_vectors = []

WIDTH_OF_BIN = 0.01
hash_buckets = 800

def normalize_intensity():
    """Normalize intensities across all spectra to range [0,1]"""
    # Collect all intensities from all spectra
    all_intensities = []
    for ms2 in spectra_to_compare:
        all_intensities.extend(ms2.intensity)
    
    max_int = max(all_intensities)
    min_int = min(all_intensities)
    
    def normalize_formula(intensity_array):
        res = []
        for intensity in intensity_array:
            int = (intensity - min_int) / (max_int - min_int)
            res.append(int)
        return res
    # Create normalized spectra tuples (mz, normalized_intensity)
    normalized_spectra = []
    for ms2 in spectra_to_compare:
        normalized_intensities = normalize_formula(ms2.intensity)
        # Create tuple of (mz_array, normalized_intensity_array)
        normalized_spectrum = (ms2.mz, normalized_intensities)
        normalized_spectra.append(normalized_spectrum)
    
    return normalized_spectra

# Get normalized data
normalized_spectra_tuples = normalize_intensity()

# Mutate spectra_to_compare to use normalized data
spectra_to_compare = [
    type('NormalizedSpectrum', (), {
        'mz': mz_array, 
        'intensity': intensity_array
    })() 
    for mz_array, intensity_array in normalized_spectra_tuples
]


def create_sparse_map(mz_array, intensity_array): # same as our code above.
    """Convert spectrum to sparse map representation"""
    sparse_map = {}
    for mz, intensity in zip(mz_array, intensity_array):
        idx = int(mz // WIDTH_OF_BIN)
        sparse_map[idx] = intensity
    return sparse_map

def sparse_map_to_hash_vector(sparse_map, num_buckets=hash_buckets):
    """Convert sparse map to hash vector"""
    hash_vec = [0] * num_buckets
    for sparse_idx, intensity in sparse_map.items():
        byte_representation = int(sparse_idx).to_bytes(8, 'little')
        bucket_idx = rapidhash(byte_representation) % num_buckets
        hash_vec[bucket_idx] += intensity
    return hash_vec
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors (returns value between 0 and 1)"""
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    
    # Calculate dot product
    dot_prod = np.dot(vec1, vec2)
    
    # Calculate magnitudes
    norm1 = np.linalg.norm(vec1) # here's the "cosine" part of cosine_similarity
    norm2 = np.linalg.norm(vec2)
    
    # Avoid division by zero
    if norm1 == 0 or norm2 == 0:
        return 0.0
    
    # Cosine similarity = dot_product / (norm1 * norm2)
    return dot_prod / (norm1 * norm2)
def sparse_cosine_similarity(map1, map2):
    """Calculate cosine similarity between two sparse maps (returns value between 0 and 1)"""
    # Get all unique indices from both maps
    all_indices = set(map1.keys()) | set(map2.keys())
    
    # Convert sparse maps to dense vectors for the shared indices

    vec1 = np.array([map1.get(idx, 0.0) for idx in sorted(all_indices)])
    vec2 = np.array([map2.get(idx, 0.0) for idx in sorted(all_indices)])
    # Calculate cosine similarity using the dot_product function
    return cosine_similarity(vec1, vec2)

# Create representations for each spectrum
for spec_data in spectra_to_compare:
    sparse_map = create_sparse_map(spec_data.mz, spec_data.intensity)
    hash_vec = sparse_map_to_hash_vector(sparse_map,hash_buckets)
    
    sparse_maps.append(sparse_map)
    hash_vectors.append(hash_vec)

# Calculate similarity matrices

n_spectra = len(spectra_to_compare)
sparse_similarities = np.zeros((n_spectra, n_spectra)) # tuples
hash_similarities = np.zeros((n_spectra, n_spectra))

# Calculate pairwise similarities using cosine similarity
for i in range(n_spectra):
    for j in range(n_spectra):
        # Sparse map cosine similarities (0 to 1)
        sparse_similarities[i, j] = sparse_cosine_similarity(sparse_maps[i], sparse_maps[j])
        
        # Hash vector cosine similarities (0 to 1)  
        hash_similarities[i, j] = cosine_similarity(hash_vectors[i], hash_vectors[j])


# When you compute pairwise similarities between spectra, you get a symmetric matrix
#      Spec0  Spec1  Spec2  Spec3
# Spec0  1.0   0.8    0.3    0.6
# Spec1  0.8   1.0    0.4    0.5
# Spec2  0.3   0.4    1.0    0.7
# Spec3  0.6   0.5    0.7    1.0
#
# Using np.triu allows us to just grab the upper (right) triangle. np.tril_indices would grab the lower left.
#
# Show distance between sparse and hash similarities using cosine distance
sparse_upper = sparse_similarities[np.triu_indices(n_spectra, k=1)]
hash_upper = hash_similarities[np.triu_indices(n_spectra, k=1)]

import numpy as np
from scipy.spatial import distance
import pandas as pd

# Calculate cosine distance between the similarity matrices
cosine_similarity_preservation = 1- distance.cosine(sparse_upper, hash_upper)
correlation = np.corrcoef(sparse_upper, hash_upper)

print(f"\nSimilarity Preservation Analysis:")
print(f"Cosine similarity between sparse and hash similarities: {cosine_similarity_preservation:.3f}")
print(f"This shows how well the hash representation preserves the similarity structure")
print(f"A similarity close to 1 indicates good similarity preservation")
print("shape of correlation matrix:", correlation.shape, "shape of other arrays:", sparse_similarities.shape, hash_similarities.shape)

# Create multiple cleaner visualizations instead of one overwhelming scatter plot
fig, axes = plt.subplots(1, 1, figsize=(10, 6))
# Difference histogram - shows how well similarities are preserved
differences = hash_upper - sparse_upper
axes.hist(differences, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes.axvline(0, color='red', linestyle='--', linewidth=2, label='Perfect preservation')
axes.set_xlabel('Hash Similarity - Sparse Similarity')
axes.set_ylabel('Frequency')
axes.set_title(f'Similarity Preservation Errors\nMean: {np.mean(differences):.4f}, Std: {np.std(differences):.4f}')
axes.legend()
axes.set_xlim(-0.1, 1.0)
axes.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\nDetailed Statistics:")
print(f"Number of pairwise comparisons: {len(sparse_upper):,}")
print(f"Sparse similarities range: {np.min(sparse_upper):.3f} to {np.max(sparse_upper):.3f}")
print(f"Hash similarities range: {np.min(hash_upper):.3f} to {np.max(hash_upper):.3f}")
print(f"Mean absolute difference: {np.mean(np.abs(differences)):.4f}")
print(f"Pearson correlation coefficient: {correlation[0,1]:.4f}")

# Convert sparse similarities to DataFrame for better display
sparse_df = pd.DataFrame(sparse_similarities)
sparse_df.index.name = 'Spectrum'
sparse_df.columns.name = 'Spectrum'

# Convert hash similarities to DataFrame for better display  
hash_df = pd.DataFrame(hash_similarities)
hash_df.index.name = 'Spectrum'
hash_df.columns.name = 'Spectrum'

import seaborn as sns

# Perform hierarchical clustering on the sparse (unhashed) similarities
# Convert similarity to distance (1 - similarity) for clustering
sparse_distances = 1 - sparse_similarities

# Create clustermaps with the same ordering
print("Clustering based on unhashed similarities, showing both matrices with same ordering:")

# First clustermap: sparse similarities (this determines the ordering)
fig1 = sns.clustermap(1-sparse_df, method="average", cmap="viridis", 
                      figsize=(10, 8), cbar_pos=(0.02, 0.8, 0.03, 0.18))
fig1.figure.suptitle('Unhashed (Sparse) Matrix Similarities\n(Hierarchical Clustering)', y=0.95)

# grab ordering from fig 1
row_order = fig1.dendrogram_row.reordered_ind
col_order = fig1.dendrogram_col.reordered_ind

plt.show()

# Reorder the hash_df according to the clustering results from fig 1
hash_df_reordered = hash_df.iloc[row_order, col_order]

fig2 = sns.clustermap(1-hash_df_reordered, method="average", cmap="viridis",
                      row_cluster=False, col_cluster=False,  # Don't re-cluster, use existing order
                      figsize=(10, 8), cbar_pos=(0.02, 0.8, 0.03, 0.18))
fig2.figure.suptitle('Hashed Matrix Similarities\n(Same ordering as unhashed clustering)', y=0.95)

plt.show()

## Spectral Hashing Example, Start to Finish

### Set-Up/Background

Below is the ion ladder for the peptide: AVVQDPALKPLALVYGEATSR.

In [ ]:
make_ion_ladder('AVVQDPALKPLALVYGEATSR')

Let's look at how this ion ladder can be plotted as a "spectrum." The purpose of this is to visualize the spread of m/z values. The intensity values here are meaningless.

In [ ]:
b_mz = [
    72.044114, 171.112528, 270.180942, 398.239520, 513.266463,
    610.319227, 681.356341, 794.440405, 922.535368, 1019.588132,
    1132.672196, 1203.709310, 1316.793374, 1415.861788, 1578.925108,
    1635.946572, 1764.989165, 1836.026279, 1937.073958, 2024.105986
]

y_mz = [
    2127.179698, 2028.111284, 1929.042870, 1800.984292, 1685.957349,
    1588.904585, 1517.867471, 1404.783407, 1276.688444, 1179.635680,
    1066.551616, 995.514502, 882.430438, 783.362024, 620.298704,
    563.277240, 434.234647, 363.197533, 262.149854, 175.117826
]

fig, ax = plt.subplots(figsize=(8, 4), dpi=100)

# Equal heights (e.g., 1.0)
height = 0.6
ax.vlines(b_mz, 0, height, colors='#1976D2', linewidth=1.5, label='b-ions')
ax.vlines(y_mz, 0, height, colors='#D32F2F',  linewidth=1.5, label='y-ions')

ax.set_ylim(0, 1.1)
ax.set_xlabel("m/z")
ax.set_ylabel("Intensity (arbitrary)")
ax.set_title("Theoretical ion ladder for AVVQDPALKPLALVYGEATSR")
ax.legend(frameon=False)
ax.grid(True, axis='y', alpha=0.25)

# b labels: b1, b3, b5, ...
for i, x in enumerate(b_mz, start=1):
    if i % 2 == 1:
        ax.text(x, height*1.02, f"b{i}", rotation=90, ha="center", va="bottom", fontsize=8)

# y_mz is descending (y20..y1). Convert to ion number j = y#
n_y = len(y_mz)
for i, x in enumerate(y_mz, start=1):
    j = n_y - i + 1   # y ion number at this m/z
    if j % 2 == 1:    # y1, y3, y5, ...
        ax.text(x, height*1.02, f"y{j}", rotation=90, ha="center", va="bottom", fontsize=8)

plt.show()

Now that we've visualized this theoretical ion ladder as a "spectrum," let's plot a real spectrum and use the ion ladder to annotate it. The spectrum plotted below is the unmodified AVVQDPALKPLALVYGEATSR peptide.

In [ ]:
plot_MS2(get_MS2_object(full_calibrated_mzml_path, 9970, peptide = 'AVVQDPALKPLALVYGEATSR')) # Scan 9970 is the unmodified spectrum for sequence AVVQDPALKPLALVYGEATSR

Although there is inevitable noise that deviates from the theoretical ion ladder, this spectrum overall aligns really well with the ladder. In other words, **a significant proportion of this spectrum's total intensity is accounted for by the theoretical ion ladder.** Let's now look at a modified version of AVVQDPALKPLALVYGEATSR.

In [ ]:

seq = 'AVVQDPALKPLALVYGEATSR'

spec_left  = get_MS2_object(full_calibrated_mzml_path, 9970, peptide=seq)
spec_right = get_MS2_object(full_calibrated_mzml_path, 8090, peptide=seq)

# Make two panels sharing axes so scales match
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120, sharex=True, sharey=True)
axes[0].set_xlabel("m/z")
axes[0].set_ylabel("Intensity")

# Left panel = unmodified = scan 9970
sup.spectrum(spec_left, ax=axes[0], grid=True)
axes[0].set_title("Unmodified (Scan 9970)")

# Right panel = modified = scan 8090
sup.spectrum(spec_right, ax=axes[1], grid=True)
axes[1].set_title("Modified (Scan 8090)")

fig.suptitle("AVVQDPALKPLALVYGEATSR — Unmodified vs. Modified", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.94])  # leave space at the top for the suptitle
plt.show()

We can repeat that process using an unmodified and modified spectrum from each of the 2 other peptides in our mzml file: IITHPNFNGNTLDNDIMLIK and RMVNNGHSFNVEYDDSQDK.

In [ ]:

seq = 'IITHPNFNGNTLDNDIMLIK'
spec_left  = get_MS2_object(full_calibrated_mzml_path, 7567, peptide=seq)
spec_right = get_MS2_object(full_calibrated_mzml_path, 8616, peptide=seq)

# Make two panels sharing axes so scales match
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120, sharex=True, sharey=True)
axes[0].set_xlabel("m/z")
axes[0].set_ylabel("Intensity")

# Left panel = unmodified = scan 7567
sup.spectrum(spec_left, ax=axes[0], grid=True)
axes[0].set_title("Unmodified (Scan 7567)")

# Right panel = modified = scan 8616
sup.spectrum(spec_right, ax=axes[1], grid=True)
axes[1].set_title("Modified (Scan 8616)")

fig.suptitle("IITHPNFNGNTLDNDIMLIK — Unmodified vs. Modified", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.94])  # leave space at the top for the suptitle
plt.show()

In [ ]:

seq = 'RMVNNGHSFNVEYDDSQDK'

spec_left  = get_MS2_object(full_calibrated_mzml_path, 3864, peptide=seq)
spec_right = get_MS2_object(full_calibrated_mzml_path, 4022, peptide=seq)

# Make two panels sharing axes so scales match
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120, sharex=True, sharey=True)
axes[0].set_xlabel("m/z")
axes[0].set_ylabel("Intensity")

# Left panel = unmodified = scan 3864
sup.spectrum(spec_left, ax=axes[0], grid=True)
axes[0].set_title("Unmodified (Scan 3864)")

# Right panel = modified = scan 4022
sup.spectrum(spec_right, ax=axes[1], grid=True)
axes[1].set_title("Modified (Scan 4022)")

fig.suptitle("RMVNNGHSFNVEYDDSQDK — Unmodified vs. Modified", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.94])  # leave space at the top for the suptitle
plt.show()

There are two "trends" you might be noticing by now: <br></br>
1. Generally, when a spectrum (modified or unmodified) is annotated with the theoretical ion ladder for it's associated peptide, there is a significant proportion of that spectrum's intensity that is accounted for by the ion ladder.
2. Unmodified spectra better "match" or are better "accounted for" by the peptide's theoretical ion ladder than modified spectra. But there is not a significant difference.
<br></br>
But what if we were to use the ion ladder of one peptide to annotate the spectrum of a different peptide? Let's try using the theoretical ion ladder of AVVQDPALKPLALVYGEATSR to annotate the spectrum of a modified RMVNNGHSFNVEYDDSQDK peptide. We'll plot that on the right panel. On the left, we'll plot the spectrum of a modified AVVQDPALKPLALVYGEATSR spectrum and annotate it with the AVVQDPALKPLALVYGEATSR ion ladder (just as we did above). In this case, we are plotting spectra from **2 different peptides** and annotating them with **1 ion ladder.**

In [ ]:
seq = 'AVVQDPALKPLALVYGEATSR'

spec_left  = get_MS2_object(full_calibrated_mzml_path, 8090, peptide=seq)
spec_right = get_MS2_object(full_calibrated_mzml_path, 4022, peptide=seq)

# Make two panels sharing axes so scales match
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120, sharex=True, sharey=True)
axes[0].set_xlabel("m/z")
axes[0].set_ylabel("Intensity")

# Left panel = modified AVVQDPALKPLALVYGEATSR = scan 8090
sup.spectrum(spec_left, ax=axes[0], grid=True)
axes[0].set_title("Modified AVVQDPALKPLALVYGEATSR (scan 8090)")

# Right panel = modified RMVNNGHSFNVEYDDSQDK = scan 4022
sup.spectrum(spec_right, ax=axes[1], grid=True)
axes[1].set_title("Modified RMVNNGHSFNVEYDDSQDK (Scan 4022)")

fig.suptitle("Annotation from AVVQDPALKPLALVYGEATSR Ion Ladder", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.94])  # leave space at the top for the suptitle
plt.show()

Let's do that once more, but this time annotating the spectrum of the modified RMVNNGHSFNVEYDDSQDK peptide using the theoretical ion ladder of RMVNNGHSFNVEYDDSQDK (on the left) and AVVQDPALKPLALVYGEATSR (on the right). In this case, we are plotting **1 spectrum** and annotating it with **2 different ion ladders.**

In [ ]:
seq_1= 'RMVNNGHSFNVEYDDSQDK'
seq_2 = 'AVVQDPALKPLALVYGEATSR'

spec_left  = get_MS2_object(full_calibrated_mzml_path, 4022, peptide=seq_1)
spec_right = get_MS2_object(full_calibrated_mzml_path, 4022, peptide=seq_2)

# Make two panels sharing axes so scales match
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120, sharex=True, sharey=True)
axes[0].set_xlabel("m/z")
axes[0].set_ylabel("Intensity")

# Left panel = RMVNNGHSFNVEYDDSQDK Annotation, Scan 4022
sup.spectrum(spec_left, ax=axes[0], grid=True)
axes[0].set_title("RMVNNGHSFNVEYDDSQDK Annotation")

# Right panel = AVVQDPALKPLALVYGEATSR Annotation, scan 4022
sup.spectrum(spec_right, ax=axes[1], grid=True)
axes[1].set_title("AVVQDPALKPLALVYGEATSR Annotation")

fig.suptitle("Modified RMVNNGHSFNVEYDDSQDK Spectrum", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.94])  # leave space at the top for the suptitle
plt.show()

Now, you're probably beginning to notice more meaningful trends. In the first example, where we annotated two different spectra (from different peptides) with the same ion ladder, a much greater proportion of the "matching" spectrum's intensity was accounted for by the ion ladder than of the other spectrum's intensity. In the second example, where we annotated the same spectrum using two different ion ladders (one belonging to the same peptide, and one not), a much greater proportion of the spectrum (annotated by it's associated peptide)'s intensity was accounted for by the ion ladder than of the spectrum (annoted by the other peptide)'s intensity. **Let's begin the binning process:**

In [ ]:

import importlib
import SpectrumWithTransformations as swt_module
import SpectrumWithTransformations
from spectrum_utils.proforma import Modification

importlib.reload(swt_module)

# This function should read in an mzml file and return an object of type SpectrumWithTransformations
# Based off of get_MS2_object from Sam Payne lesson 4
def get_SWT_object(
    mzml_path: str,
    scan_number: int,
    full_sequence = None,
) -> "SpectrumWithTransformations":
    
    index = scan_number -1 #scan_number is 1-based, index is 0-based
    with mzml.MzML(mzml_path, use_index=True) as reader: #use_index=True allows us to avoid reading through the entire mzml file
        selected_spectrum = reader.get_by_index(index)
    # Test to see if we accessed the correct scan: PASSED!
    # precursor_mz = selected_spectrum['precursorList']['precursor'][0]['isolationWindow']['isolation window target m/z']
    # print(precursor_mz)
    

    # This finds the cooresponding values in the .mzml file to create our MS2 for a given scan (see the params)
    spectrum_id = selected_spectrum['id']
    retention_time = selected_spectrum['scanList']['scan'][0]['scan start time']
    precursor_mz = selected_spectrum['precursorList']['precursor'][0]['isolationWindow']['isolation window target m/z']
    precursor_charge = int(selected_spectrum['precursorList']['precursor'][0]['selectedIonList']['selectedIon'][0]['charge state'])
    mz_array = np.asarray(selected_spectrum['m/z array'])
    intensity_array = np.asarray(selected_spectrum['intensity array'])
    
    swt_object = SpectrumWithTransformations.SpectrumWithTransformations(
        identifier=spectrum_id,
        scan_number=scan_number,
        precursor_mz=precursor_mz,
        precursor_charge=precursor_charge,
        mz_array=mz_array,
        intensity_array=intensity_array,
        retention_time=retention_time,
        annotation_dictionary=None,
        binned_mz=None,
        hashed_mz=None,
    )

    if full_sequence:
        swt_object = swt_object.annotate_proforma(
            proforma_str = full_sequence,
            fragment_tol_mass = 0.01, # We consider two peaks (actual and theoretical) "equivalent" if they are within +/- 0.01 Da
            fragment_tol_mode = 'Da',
            ion_types = 'by',
            max_ion_charge = max(1, precursor_charge - 1)
        )
    return swt_object

In [ ]:
# from spectrum_utils.proforma import Modification
# Example with Scan 8090
# mod_1 = Modification(  
    # mass = 0.9848,
    # position = 4,
    # label = 'Deamidation'
# )
# modifications_list = [mod_1]

scan_8090 = get_SWT_object(
    mzml_path=full_calibrated_mzml_path,
    scan_number = 8090,
    full_sequence = 'AVVQ[Deamidated]DPALKPLALVYGEATSR',
)

plot_MS2(scan_8090, title='Scan 8090: Original Spectrum')

In [ ]:
WIDTH_OF_BIN = 0.01
def to_idx(num):
    return int(num // WIDTH_OF_BIN)

# Bin the mz
scan_8090.binned_mz = np.empty_like(scan_8090.mz, dtype=int)
for i in range(len(scan_8090.mz)):
    scan_8090.binned_mz[i] = to_idx(scan_8090.mz[i])

# Create a binned_spectrum SWT object for plotting purposes only
binned_spectrum = get_SWT_object(
    mzml_path=full_calibrated_mzml_path,
    scan_number = 8090,
    full_sequence = 'AVVQ[Deamidated]DPALKPLALVYGEATSR',
)
for i in range (len(binned_spectrum.mz)):
    binned_spectrum.mz[i] = scan_8090.binned_mz[i] #Re-writing the mz_array with the binned mz values

# Plot the binned spectrum
plot_MS2(binned_spectrum, 'Scan 8090: Binned Spectrum')


In [ ]:
hash_buckets = 800  # Target dimensionality

def hasher(num: int) -> int:
    """
    Hash function that maps sparse indices to a fixed number of buckets.

    Input: Large sparse index (e.g., 12910)
    Output: Small bucket index (0 to 399)
    """
    # Convert integer to bytes for hashing (rapidhash expects byte input)
    byte_representation = int(num).to_bytes(8, 'little')
    # Hash and mod to get bucket index in range [0, hash_buckets-1]
    return rapidhash(byte_representation) % hash_buckets

# Set-up
hashed_mz = []
hashed_intensity = []
hash_matrix = [0] * hash_buckets
mz_intensity_map = {}
for i, mz in enumerate(scan_8090.mz):
    mz_intensity_map[to_idx(mz)] = scan_8090.intensity[i]

# Hash the mz and add the intensities as we go
for sparse_idx, intensity in mz_intensity_map.items():
    bucket_idx = hasher(sparse_idx)
    hash_matrix[bucket_idx] += intensity
    hashed_mz.append(bucket_idx)
    hashed_intensity.append(hash_matrix[bucket_idx])

# Update the hashed mz and intensities
scan_8090.hashed_mz = hashed_mz
scan_8090.hashed_intensity = hashed_intensity

# Create a hashed_spectrum SWT object for plotting purposes only
hashed_spectrum = get_SWT_object(
    mzml_path=full_calibrated_mzml_path,
    scan_number = 8090,
    full_sequence = 'AVVQ[Deamidated]DPALKPLALVYGEATSR',
)
for i in range (len(hashed_spectrum.mz)):
    hashed_spectrum.mz[i] = scan_8090.hashed_mz[i] #Re-writing the mz_array with the hashed mz values
    hashed_spectrum.intensity[i] = scan_8090.hashed_intensity[i] #Re-writing the intensity_array with the summed intensity values

# Plot the hashed spectrum
plot_MS2(hashed_spectrum, 'Scan 8090: Hashed Spectrum')


### Side by side
<img src="Scan8090_Original.png" width="400" height="300">
<img src="Scan8090_Binned.png" width="400" height="300">
<img src="Scan8090_Hashed.png" width="400" height="300">